# MyHumNews — XLM-R hyperparameter sweep (Kaggle)The linear arm was swept to exhaustion: C, max_features, min_df, analyzer,n-gram range, max_df and four model classes. XLM-R has had **one** run atlibrary defaults (`lr=2e-5`, `pos_weight` capped at 50, 3 epochs) and alreadywins 15 of 19 labels. This notebook closes that gap so the comparison istuned-against-tuned.## Two rules this notebook does not break1. **Selection happens on validation, never on test.** Every number here is a   val macro-F1. The test set is not loaded, not embedded, not scored. The   winning config goes back to `dvb_xlmr_colab.ipynb`, which scores test once.   (That notebook used to pick its best *epoch* by test macro-F1 — a smaller   version of the same mistake. It now selects on val.)2. **The protocol is inherited, not re-implemented.** Split, label set,   `prf` and `best_threshold` come from the same data file and the same code   as every earlier number, so a sweep result is comparable to the 0.643   linear baseline without any re-derivation.## BudgetKaggle gives ~30 GPU-hours/week with sessions of 9–12 hours. The sweep issized to fit one week with room to spare:| phase | configs | data | epochs | eval | cost each | total ||---|---|---|---|---|---|---|| A — learning rate | 4 | 40% subset | 2 | val | ~1.3 h | ~5.2 h || B — pos_weight cap | 2 | 40% subset | 2 | val | ~1.3 h | ~2.6 h || C — final, full data | 1 | 100% | 3 | val + test **once** | ~4.8 h | ~4.8 h |Phases A and B run here. Phase C runs in `dvb_xlmr_colab.ipynb` with thewinning values, because that notebook already has per-epoch checkpointing andresume and has been exercised on a real run.**Three things make the subset phases cheap.** The subset is 40% of train;only 2 epochs run, which is enough to rank learning rates; and test is neverscored, which alone removes 80% of the evaluation cost (24,539 test rowsagainst 5,622 val rows).**Gradient checkpointing is off here.** It was faster on the laptop onlybecause 16 GB of unified memory was under pressure. On a T4 with fp16 atbatch 16 the model needs roughly 8 GB, so checkpointing is pure overhead —about 1.4x on step time.## ResumingKaggle will cut you off mid-sweep. Every finished config is appended to`/kaggle/working/sweep.json` and skipped on the next run, so restartingcosts at most the one config that was interrupted. No model weights aresaved during the sweep — only scores — because a 20 GB working directorycannot hold six 3 GB checkpoints and re-running one config is cheaper thanstoring them all.

In [ ]:
!nvidia-smi || echo 'NO GPU — Settings -> Accelerator -> GPU before going further'

In [ ]:
!pip -q install -U 'transformers>=4.44' 2>&1 | tail -2
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
assert torch.cuda.is_available(), 'no CUDA — switch the accelerator on'
print('gpu', torch.cuda.get_device_name(0),
      '%.0f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print('n_gpu', torch.cuda.device_count())

## Data`locate()` checks the places Kaggle and Colab actually put files. If thedataset is not attached, it falls back to pulling the two files straight fromDrive — that download runs at datacenter speed and never touches your laptop.

In [ ]:
import glob, os

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
SWEEP = os.path.join(WORK, 'sweep.json')
DATA_NAME, META_NAME = 'dvb_xlmr_data.jsonl.gz', 'dvb_xlmr_meta.json'

# Drive file ids, used only if the Kaggle dataset is not attached.
GDRIVE = {DATA_NAME: '13ciWqs1ZQxZq44bTs75mqHK2xz-tMUwU',
          META_NAME: '1OaU7GaPWI28L8Ng5BgiWo9Mw82Y4yuXB'}


def locate(name):
    for c in [os.path.join(WORK, name), name,
              os.path.join('/kaggle/working', name)]:
        if os.path.exists(c):
            return c
    hit = glob.glob('/kaggle/input/**/' + name, recursive=True)
    if hit:
        return hit[0]
    # Not attached as a dataset -> pull it from Drive.
    dest = os.path.join(WORK, name)
    print('%s not found locally; downloading from Drive...' % name, flush=True)
    os.system('pip -q install -U gdown')
    os.system('gdown -q %s -O %s' % (GDRIVE[name], dest))
    if not os.path.exists(dest):
        raise FileNotFoundError(
            '%s could not be found or downloaded. Attach it as a Kaggle '
            'dataset, or check that the Drive file is link-shared.' % name)
    return dest


DATA, META = locate(DATA_NAME), locate(META_NAME)
print('data %s  (%.1f MB)' % (DATA, os.path.getsize(DATA) / 1e6))
print('meta %s' % META)

In [ ]:
import gzip, json, numpy as np

meta = json.load(open(META))
LABELS = meta['labels']
L = {l: j for j, l in enumerate(LABELS)}
SEED = meta['seed']

# Only train and val are read. Test is deliberately left on disk: nothing
# in this notebook can select on it, even by accident.
rows = {'train': [], 'val': []}
with gzip.open(DATA, 'rt', encoding='utf-8') as fh:
    for line in fh:
        d = json.loads(line)
        if d['split'] in rows:
            rows[d['split']].append((d['text'], d['labels']))


def ymat(rs):
    M = np.zeros((len(rs), len(LABELS)), dtype=np.float32)
    for r, (_, ls) in enumerate(rs):
        for l in ls:
            M[r, L[l]] = 1.0
    return M


Xtr = [t for t, _ in rows['train']]; Ytr = ymat(rows['train'])
Xva = [t for t, _ in rows['val']];   Yva = ymat(rows['val'])
assert len(Xtr) == meta['splits']['train'], 'SPLIT MISMATCH — stop'
assert len(Xva) == meta['splits']['val'],   'SPLIT MISMATCH — stop'
print('train %d  val %d   (test %d not loaded, on purpose)'
      % (len(Xtr), len(Xva), meta['splits']['test']))
print('labels %d' % len(LABELS))

## MetricsCopied unchanged from `dataset/tune_dvb.py`, which produced every linear-armnumber. The sweep reports **val** macro-F1, where the threshold for each labelis fitted on the same val slice it scores. That is optimistic in absolute terms— it is a fit, not a held-out estimate — but it is the same optimism for everyconfig, so it ranks them correctly. Only the Phase C test number is unbiased,and that is the one reported anywhere else.

In [ ]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def best_threshold(scores, gold):
    best_t, best_f = 0.0, -1.0
    for t in np.unique(np.round(scores, 2)):
        pred = scores >= t
        f = prf(int((pred & (gold == 1)).sum()),
                int((pred & (gold == 0)).sum()),
                int((~pred & (gold == 1)).sum()))[2]
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t


def macro_val(SV, Yv, Yt):
    # Same label filter as everywhere else: a label needs positives in both
    # the training data and the slice being scored.
    f1s = []
    for j in range(len(LABELS)):
        if Yt[:, j].sum() == 0 or Yv[:, j].sum() == 0:
            continue
        g = Yv[:, j].astype(int)
        thr = best_threshold(SV[:, j], g)
        pred = SV[:, j] >= thr
        gold = g == 1
        f1s.append(prf(int((pred & gold).sum()), int((pred & ~gold).sum()),
                       int((~pred & gold).sum()))[2])
    return sum(f1s) / len(f1s)

## One training run`run_one` builds a fresh model, trains it, and returns val macro-F1 per epoch.It frees the GPU on the way out so six configs can run in one session.`SUBSET` is applied with a fixed seed, so every config sees the *same* subset.The labels are rare enough that a different draw per config would add noisebigger than the effect being measured.

In [ ]:
import gc, time, random
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL   = 'xlm-roberta-base'
MAX_LEN = 512          # XLM-R's ceiling; not a tunable
BATCH   = 16
EVAL_B  = 64

tok = AutoTokenizer.from_pretrained(MODEL)
dev = 'cuda'


class DS(Dataset):
    def __init__(self, xs, y=None): self.xs, self.y = xs, y
    def __len__(self): return len(self.xs)
    def __getitem__(self, i):
        return self.xs[i], (None if self.y is None else self.y[i])


def collate(batch):
    enc = tok([b[0] for b in batch], truncation=True, max_length=MAX_LEN,
              padding=True, return_tensors='pt')
    if batch[0][1] is None:
        return enc, None
    return enc, torch.tensor(np.stack([b[1] for b in batch]))


def subset(frac):
    if frac >= 1.0:
        return Xtr, Ytr
    rng = random.Random(SEED)                 # same draw for every config
    idx = rng.sample(range(len(Xtr)), int(len(Xtr) * frac))
    return [Xtr[i] for i in idx], Ytr[idx]


def run_one(lr, pos_cap, epochs, frac, tag):
    xs, ys = subset(frac)
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(LABELS),
        problem_type='multi_label_classification').to(dev)

    if pos_cap is None:
        lossf = torch.nn.BCEWithLogitsLoss()
    else:
        n_pos = ys.sum(axis=0)
        pw = np.minimum(np.where(n_pos > 0,
                                 (len(xs) - n_pos) / np.maximum(n_pos, 1),
                                 1.0), pos_cap)
        lossf = torch.nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(pw, dtype=torch.float32, device=dev))

    dl = DataLoader(DS(xs, ys), batch_size=BATCH, shuffle=True,
                    collate_fn=collate, num_workers=2, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=epochs * len(dl), pct_start=0.1)
    bf16 = torch.cuda.is_bf16_supported()
    amp_dt = torch.bfloat16 if bf16 else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=not bf16)

    @torch.no_grad()
    def infer(txt):
        model.eval(); out = []
        for enc, _ in DataLoader(DS(txt), batch_size=EVAL_B, shuffle=False,
                                 collate_fn=collate, num_workers=2):
            enc = {k: v.to(dev) for k, v in enc.items()}
            with torch.autocast('cuda', dtype=amp_dt):
                out.append(model(**enc).logits.float().cpu().numpy())
        model.train()
        return np.concatenate(out)

    hist, t0 = [], time.time()
    for ep in range(1, epochs + 1):
        model.train(); run = 0.0
        for step, (enc, yb) in enumerate(dl, 1):
            enc = {k: v.to(dev) for k, v in enc.items()}
            with torch.autocast('cuda', dtype=amp_dt):
                loss = lossf(model(**enc).logits, yb.to(dev))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
            run += loss.item()
            if step % 200 == 0:
                el = (time.time() - t0) / 60
                print('    %s ep%d %d/%d  loss %.4f  %.0fm'
                      % (tag, ep, step, len(dl), run / step, el), flush=True)
        v = macro_val(infer(Xva), Yva, ys)
        hist.append(v)
        print('    %s epoch %d  VAL macro-F1 %.4f  [%.0fm]'
              % (tag, ep, v, (time.time() - t0) / 60), flush=True)

    del model, opt, sched, dl, lossf
    gc.collect(); torch.cuda.empty_cache()
    return hist, (time.time() - t0) / 60

## The sweepPhase A moves the learning rate with everything else held at the values thebaseline run used. Phase B then takes the winning learning rate and moves the`pos_weight` cap, which is the knob most likely to matter here: the baselinerun reached **precision 0.049 at recall 0.909** on COVID19, which is whatover-weighting a rare label looks like. Values 50 (baseline), 20, and None(no weighting at all) bracket it.Phase B depends on Phase A's answer, so the two are not one grid — runningthem as a 4x3 grid would cost 12 configs instead of 6 for no extrainformation.

In [ ]:
PHASE_A = [dict(lr=lr, pos_cap=50.0) for lr in (1e-5, 2e-5, 3e-5, 5e-5)]
PHASE_B_CAPS = [20.0, None]      # 50.0 already covered by phase A

EPOCHS_SEARCH = 2
FRAC          = 0.40


def load_sweep():
    if os.path.exists(SWEEP):
        return json.load(open(SWEEP))
    return {'runs': []}


def save_sweep(s):
    json.dump(s, open(SWEEP + '.tmp', 'w'), ensure_ascii=False, indent=1)
    os.replace(SWEEP + '.tmp', SWEEP)      # atomic


def key(lr, pos_cap):
    return 'lr%g_cap%s' % (lr, pos_cap)


def already(s, lr, pos_cap):
    return next((r for r in s['runs'] if r['key'] == key(lr, pos_cap)), None)


def do(lr, pos_cap):
    s = load_sweep()
    hit = already(s, lr, pos_cap)
    if hit:
        print('skip %s  (done: val %.4f)' % (hit['key'], hit['val_macro_f1']))
        return hit
    tag = key(lr, pos_cap)
    print('\n=== %s ===' % tag, flush=True)
    hist, mins = run_one(lr, pos_cap, EPOCHS_SEARCH, FRAC, tag)
    rec = {'key': tag, 'lr': lr, 'pos_cap': pos_cap,
           'epochs': EPOCHS_SEARCH, 'frac': FRAC, 'model': MODEL,
           'max_len': MAX_LEN, 'batch': BATCH,
           'val_macro_f1_by_epoch': [round(h, 4) for h in hist],
           'val_macro_f1': round(max(hist), 4),
           'minutes': round(mins, 1),
           'gpu': torch.cuda.get_device_name(0)}
    s = load_sweep(); s['runs'].append(rec); save_sweep(s)
    print('=== %s  val %.4f  (%.0f min) ===' % (tag, rec['val_macro_f1'], mins))
    return rec

In [ ]:
# PHASE A — learning rate. Safe to interrupt; finished configs are skipped.
for cfg in PHASE_A:
    do(cfg['lr'], cfg['pos_cap'])

s = load_sweep()
a = [r for r in s['runs'] if r['pos_cap'] == 50.0]
a.sort(key=lambda r: -r['val_macro_f1'])
print('\nphase A, by val macro-F1')
for r in a:
    print('  lr %-8g  val %.4f  %s  (%.0f min)'
          % (r['lr'], r['val_macro_f1'], r['val_macro_f1_by_epoch'],
             r['minutes']))
BEST_LR = a[0]['lr'] if a else 2e-5
print('\nbest lr: %g' % BEST_LR)

In [ ]:
# PHASE B — pos_weight cap at the winning learning rate.
for cap in PHASE_B_CAPS:
    do(BEST_LR, cap)

s = load_sweep()
b = [r for r in s['runs'] if r['lr'] == BEST_LR]
b.sort(key=lambda r: -r['val_macro_f1'])
print('\nphase B, at lr %g' % BEST_LR)
for r in b:
    print('  cap %-6s  val %.4f  %s' % (r['pos_cap'], r['val_macro_f1'],
                                        r['val_macro_f1_by_epoch']))
BEST_CAP = b[0]['pos_cap'] if b else 50.0
print('\nbest pos_weight cap: %s' % BEST_CAP)

## ResultThe winner's numbers are **val** macro-F1 on a 40% subset after 2 epochs. Theyare not comparable to 0.643 and must not be quoted as a result — their onlyjob is to rank configs. Phase C produces the comparable number.

In [ ]:
s = load_sweep()
rank = sorted(s['runs'], key=lambda r: -r['val_macro_f1'])
print('%-18s %10s %8s %8s' % ('config', 'val macro', 'min', 'by epoch'))
for r in rank:
    print('%-18s %10.4f %8.0f  %s' % (r['key'], r['val_macro_f1'],
                                      r['minutes'],
                                      r['val_macro_f1_by_epoch']))

w = rank[0]
base = next((r for r in s['runs'] if r['key'] == key(2e-5, 50.0)), None)
print('\n' + '=' * 62)
print('  winner   lr=%g  pos_cap=%s   val %.4f'
      % (w['lr'], w['pos_cap'], w['val_macro_f1']))
if base:
    print('  baseline lr=2e-05 pos_cap=50.0   val %.4f   (%+.4f)'
          % (base['val_macro_f1'], w['val_macro_f1'] - base['val_macro_f1']))
print('=' * 62)
print('\nPhase C: set these in dvb_xlmr_colab.ipynb cell 14 and run it on the\n'
      'full training set for 3 epochs. That notebook scores test once.')
print('  LR      = %g' % w['lr'])
print('  POS_CAP = %s' % w['pos_cap'])
print('\nIf the winner improves on the baseline by less than about 0.005, the\n'
      'defaults were already fine — say so and keep them. A sweep that finds\n'
      'nothing is a result, not a failure.')